<a href="https://colab.research.google.com/github/zabdielVR/IA_with_collab/blob/main/Copy_of_Hands_On_Fundamentos_de_LLMs_con_Modelos_Llama_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [22]:
# Instalar cliente de Groq y leer API key desde Colab Secrets

!pip install groq -q

import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [23]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)

prompt = "¿Cuál es la diferencia entre un galaxia y un universo?"
# prompt = "Explica en un párrafo qué hace un router doméstico"

print(prompt)

¿Cuál es la diferencia entre un galaxia y un universo?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [24]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada
#se susitituye el modelo ya el modelo ya no esta disponible en groq para cuentas de pago

response_ligero = client.chat.completions.create(

    model="openai/gpt-oss-20b",
    #model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

**Galaxia**  
- Es una agrupación gravitacional de estrellas, gas, polvo, y materia oscura.  
- Una galaxia típica contiene desde unas pocas miles de millones hasta más de 200 billones de estrellas.  
- Su tamaño varía: el tamaño típico de una galaxia espiral como la Vía Láctea es de unos 30 000 años luz en diámetro.  
- Dentro de una galaxia viven sistemas planetarios, nebulosas, cúmulos estelares, etc.  
- La galaxia es una estructura *limitada* dentro del espacio, con un borde definido (en términos de densidad o de la región donde la gravedad domina sobre la expansión del universo).

**Universo**  
- Es el conjunto de todo lo que existe: toda la materia y energía, el espacio‑tiempo, las leyes físicas.  
- Incluye todas las galaxias, cúmulos de galaxias, filamentos, vacíos, y la propia energía oscura que permea el espacio.  
- La escala del universo es inmensamente mayor: el diámetro del universo observable es ~93 mil millones de años luz; el universo total puede ser infinito.  
- El

In [25]:
print(response_ligero.model_dump_json(indent=2))  # para inspeccionar la respuesta completa

{
  "id": "chatcmpl-8d86c7fa-a5ac-4237-b891-55ec1f12abe4",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "**Galaxia**  \n- Es una estructura relativamente pequeña (en escala cósmica).  \n- Un conjunto de estrellas, planetas, nubes de gas y polvo, todo unido por la gravedad.  \n- La mayoría de ellas también contienen agujeros negros supermasivos en sus centros.  \n- Ejemplo: la Vía Láctea, con unos 100‑200 mil millones de estrellas y un diámetro de ~ 30 000 años luz.  \n- Se clasifica en tipos (elípticas, espirales, irregulares) y puede tener sistemas de satélites (satélites, cúmulos de estrellas, etc.).  \n\n**Universo**  \n- Es el conjunto total de todo lo que existe: todas las galaxias, cúmulos de galaxias, materia visible (estrellas, planetas, polvo, etc.) y materia “oscura” (materia oscura, energía oscura).  \n- Incluye también la radiación de fondo cósmico de microondas, los campos gravitatorios 

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [26]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

print("Tokens del prompt:", response_ligero.usage.prompt_tokens)
print("Tokens de la respuesta:", response_ligero.usage.completion_tokens)
print("Tokens totales:", response_ligero.usage.total_tokens)
# response.usage.total_tokens es lo que se factura por esta llamada

Tokens del prompt: 84
Tokens de la respuesta: 707
Tokens totales: 791


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [27]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
 #se susitituye el modelo ya que llama-3.1-8b-instant ya no esta disponible en groq para cuentas de pago

import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    #model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 1.60 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [29]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad
#se susitituye el modelo ya que llama-3.3-70b-versatile ya no esta disponible en groq para cuentas de pago


inicio = time.time()
response_grande = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    #model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s — {response_ligero.usage.total_tokens} tokens")
print(f"Modelo grande: {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

Modelo ligero: 1.60 s — 791 tokens
Modelo grande: 1.29 s — 597 tokens

Respuesta del modelo grande:
 Una **galaxia** y el **universo** son conceptos que se refieren a escalas muy diferentes del cosmos.

| Característica | Galaxia | Universo |
|----------------|---------|----------|
| **Definición** | Conjunto de estrellas, gas, polvo y materia oscura que están gravitacionalmente vinculados. | Todo lo que existe: espacio, tiempo, materia, energía y las leyes físicas que los rigen. |
| **Tamaño típico** | Entre unos pocos miles y varios cientos de miles de años‑luz de diámetro. Por ejemplo, la Vía Láctea mide ~100 000 años‑luz. | Aproximadamente 93 mil millones de años‑luz de diámetro observable, y probablemente mucho más allá (posiblemente infinito). |
| **Número** | Hay cientos de miles (según estimaciones) de galaxias dentro del universo observable. | El universo contiene **todas** esas galaxias, además de cúmulos, filamentos y vacíos. |
| **Composición** | Estrellas, nebulosas, cúmul